[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhuzaifahbidin/megat/blob/main/notebooks/02_train_tokenizer.ipynb)

# Notebook 02 — Train Custom Malay BPE Tokenizer

**What this notebook does:**
1. Loads a sample of the downloaded FineWeb2 Malay corpus
2. Trains a Byte-level BPE tokenizer with vocab size 32,000
3. Verifies that Malay words tokenize efficiently (1–2 tokens each)
4. Saves the tokenizer files to Google Drive

**Expected runtime:** 15–30 minutes (tokenizer training is CPU-bound but fast)

**Expected output:** `megat_tokenizer/` folder (vocab.json + merges.txt) saved to Drive

---
**Before this notebook:** Run `01_download_data.ipynb`  
**After this notebook:** Run `03_pretokenize_and_upload.ipynb`

---
### Why train a custom tokenizer?

The original GPT-2 tokenizer was trained on English text. When it encounters Malay:
- `masyarakat` → 5–6 tokens (it splits on unfamiliar character combinations)
- `pembangunan` → 4–5 tokens

With a Malay-trained BPE tokenizer:
- `masyarakat` → 1–2 tokens
- `pembangunan` → 1–2 tokens

This 3–4× improvement means the model's 1024-token context window covers
3–4× more actual text — dramatically improving coherence in generation.

## Step 0 — Install packages

In [ ]:
!pip install -q tokenizers

## Step 1 — Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Must match where you saved data in notebook 01
SAVE_DIR        = '/content/drive/MyDrive/megat_data'
CORPUS_FILE     = os.path.join(SAVE_DIR, 'fineweb2_malay.jsonl')
WIKI_FILE       = os.path.join(SAVE_DIR, 'malay_wikipedia.jsonl')
TOKENIZER_DIR   = os.path.join(SAVE_DIR, 'megat_tokenizer')
SAMPLE_TXT      = '/content/tokenizer_sample.txt'  # temp file in Colab RAM

os.makedirs(TOKENIZER_DIR, exist_ok=True)

assert os.path.exists(CORPUS_FILE), f'Corpus not found at {CORPUS_FILE} — run notebook 01 first!'
print(f'Corpus found: {os.path.getsize(CORPUS_FILE)/1e9:.2f} GB')

## Step 2 — Extract a training sample for the tokenizer

BPE tokenizer training works by:
1. Starting with individual bytes (256 base tokens)
2. Repeatedly finding the most frequent pair of adjacent tokens and merging them
3. Repeating until vocabulary reaches the target size (32,000)

It doesn't need the full 4M document corpus — a ~500MB sample gives the tokenizer
enough frequency statistics to learn good merges for all common Malay words and
subwords. Training on more data barely changes the resulting vocabulary.

We extract the sample as a plain `.txt` file (one document per line), which is
the format the `tokenizers` library expects.

In [ ]:
import json

# How many docs to use for tokenizer training.
# ~500K docs × average ~600 chars ≈ 300MB — more than enough.
SAMPLE_DOCS = 500_000

print(f'Extracting {SAMPLE_DOCS:,} documents to {SAMPLE_TXT} ...')

doc_count   = 0
total_chars = 0

with open(CORPUS_FILE, 'r') as src, open(SAMPLE_TXT, 'w') as dst:
    for line in src:
        text = json.loads(line)['text']
        # Write each document as a single line.
        # The tokenizer library reads line by line.
        dst.write(text.replace('\n', ' ') + '\n')
        doc_count   += 1
        total_chars += len(text)
        if doc_count >= SAMPLE_DOCS:
            break

# Also add Wikipedia if available — adds high-quality formal Malay
if os.path.exists(WIKI_FILE):
    with open(WIKI_FILE, 'r') as src, open(SAMPLE_TXT, 'a') as dst:
        for line in src:
            text = json.loads(line)['text']
            dst.write(text.replace('\n', ' ') + '\n')
    print('Wikipedia added to sample.')

sample_size_mb = os.path.getsize(SAMPLE_TXT) / 1e6
print(f'Sample ready: {doc_count:,} docs  |  {total_chars/1e6:.0f}M chars  |  {sample_size_mb:.0f} MB')

## Step 3 — Train the BPE tokenizer

We use HuggingFace's `tokenizers` library which trains BPE in minutes on CPU.

**Key settings:**
- `vocab_size=32000`: Large enough to cover Malay vocabulary well, small enough
  to not waste model capacity on rare tokens. Standard for models at this scale.
- `min_frequency=2`: A token pair must appear at least twice to be merged.
  Filters out typos and one-off character combinations.
- Special tokens: The model uses these to structure its input.
  - `<|endoftext|>`: marks document boundaries in training data
  - `<|story|>`: prompt prefix for story generation (finetuning)
  - `<|end|>`: marks end of generated story (finetuning)
  - `<|pad|>`: padding token (not used during pretraining, but needed for finetuning)

In [ ]:
import time
from tokenizers import ByteLevelBPETokenizer

# ByteLevelBPETokenizer operates on raw bytes, not Unicode characters.
# This means it can handle ANY text — Malay, Arabic loanwords, Jawi numerals,
# emojis — without needing special Unicode handling. Unknown characters are
# always representable as byte sequences, so the vocabulary is truly complete.
tokenizer = ByteLevelBPETokenizer()

print('Training BPE tokenizer...')
print(f'  Vocab size:    32,000')
print(f'  Min frequency: 2')
print(f'  Training file: {sample_size_mb:.0f} MB')
print()

t0 = time.time()

tokenizer.train(
    files=[SAMPLE_TXT],
    vocab_size=32000,
    min_frequency=2,
    special_tokens=[
        '<|endoftext|>',  # document boundary
        '<|story|>',      # story start (for finetuning prompts)
        '<|end|>',        # story end
        '<|pad|>',        # padding
    ]
)

elapsed = time.time() - t0
print(f'Training complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')

## Step 4 — Verify tokenization quality

The key metric from CLAUDE.md: tokens per word should be **1.3–1.5** on average.
The English GPT-2 tokenizer gives ~2.0+ on Malay text — we want to be well below that.

We also check individual words to confirm common Malay vocabulary encodes efficiently.

In [ ]:
print('── Individual word checks ─────────────────────────────')
print('  (target: 1–2 tokens per common word)\n')

test_words = [
    # Common vocabulary
    'masyarakat', 'pembangunan', 'kerajaan', 'perkhidmatan',
    'pendidikan', 'kesihatan', 'ekonomi', 'malaysia',
    # Morphologically complex words (prefixes + roots)
    'memperbaiki', 'perkembangan', 'pengurusan', 'perbelanjaan',
    # Short common words
    'dan', 'yang', 'untuk', 'dengan', 'pada', 'ini', 'itu',
    # Names / proper nouns
    'kuala lumpur', 'perdana menteri',
]

for word in test_words:
    enc    = tokenizer.encode(word)
    tokens = enc.tokens
    n      = len(tokens)
    flag   = '  ← check this' if n > 3 else ''
    print(f'  {word:<25} → {n} token(s): {tokens}{flag}')

In [ ]:
# Measure average tokens/word on a larger sample
import json, random

lines = []
with open(CORPUS_FILE, 'r') as f:
    for i, line in enumerate(f):
        lines.append(line)
        if i >= 10_000:
            break

sample_docs = [json.loads(l)['text'] for l in random.sample(lines, 500)]

total_words  = 0
total_tokens = 0

for doc in sample_docs:
    words        = doc.split()
    token_count  = len(tokenizer.encode(doc).ids)
    total_words  += len(words)
    total_tokens += token_count

ratio = total_tokens / total_words if total_words > 0 else 0

print(f'── Tokens-per-word ratio ───────────────────────────────')
print(f'  Sample: 500 documents')
print(f'  Total words:  {total_words:,}')
print(f'  Total tokens: {total_tokens:,}')
print(f'  Ratio:        {ratio:.2f} tokens/word')
print()
if ratio <= 1.5:
    print('  PASS — ratio is within target (≤ 1.5)')
elif ratio <= 1.8:
    print('  OK — ratio is slightly above target but acceptable')
else:
    print('  WARNING — ratio > 1.8, tokenizer may not be learning Malay well.')
    print('  Check that the training sample is actually Malay text.')

In [ ]:
# Encode and decode a full Malay sentence to verify roundtrip correctness.
# The decoded text should be identical to the original (no characters lost).

test_sentence = 'Pada suatu hari, seorang pemuda yang berani telah pergi ke hutan untuk mencari kayu.'

encoded  = tokenizer.encode(test_sentence)
decoded  = tokenizer.decode(encoded.ids)

print(f'Original: {test_sentence}')
print(f'Tokens:   {encoded.tokens}')
print(f'IDs:      {encoded.ids}')
print(f'Decoded:  {decoded}')
print()

# Check special token IDs
print('── Special token IDs ──────────────────────────────────')
for tok in ['<|endoftext|>', '<|story|>', '<|end|>', '<|pad|>']:
    tok_id = tokenizer.token_to_id(tok)
    print(f'  {tok:<20} → ID {tok_id}')
print()
print('Roundtrip OK' if test_sentence in decoded or decoded.strip() == test_sentence.strip() else 'WARNING: roundtrip mismatch')

## Step 5 — Save tokenizer to Google Drive

In [ ]:
# This saves two files:
#   vocab.json   — maps token string → token ID (the vocabulary)
#   merges.txt   — the BPE merge rules in order (pair of tokens → merged token)
#
# Both are needed to reproduce the exact tokenization at training and inference time.
# NEVER train a different tokenizer for finetuning — it must be the same one.

tokenizer.save_model(TOKENIZER_DIR)

print(f'Tokenizer saved to: {TOKENIZER_DIR}')
print(f'  Files: {os.listdir(TOKENIZER_DIR)}')
print()

# Verify we can reload it correctly
from tokenizers import ByteLevelBPETokenizer as BLBPE
reloaded = BLBPE(
    os.path.join(TOKENIZER_DIR, 'vocab.json'),
    os.path.join(TOKENIZER_DIR, 'merges.txt'),
)
test_ids = reloaded.encode('masyarakat pembangunan').ids
print(f'Reload verification: "masyarakat pembangunan" → {test_ids}')
print('Tokenizer reload OK.' if len(test_ids) < 8 else 'WARNING: reload may have failed.')

In [ ]:
# Clean up the large temp file from Colab disk (it's already in Drive)
if os.path.exists(SAMPLE_TXT):
    os.remove(SAMPLE_TXT)
    print(f'Removed temp file: {SAMPLE_TXT}')

print()
print('── Summary ────────────────────────────────────────────')
print(f'  Tokenizer: {TOKENIZER_DIR}')
print(f'  Vocab size: {tokenizer.get_vocab_size():,} tokens')
print(f'  tokens/word ratio: {ratio:.2f}')
print()
print('Next: run 03_pretokenize_and_upload.ipynb')